# Heard — Data Cleaning for Training

**Primary source**
- Kaggle: `davidgauthier/glassdoor-job-reviews`
- Role: historical labeled data for training / validation

**Supplementary source**
- Bright Data / GitHub: `luminati-io/Glassdoor-dataset-samples`
- Role: recent external evaluation and RAG/demo data
- We keep reviews dated **2022 or later** so they supplement the Kaggle dataset, which ends in 2021.

## Scope

This notebook performs **data cleaning and training preparation**:

1. Load both datasets.
2. Standardize variable names.
3. Parse dates and ratings.
4. Normalize categorical values.
5. Clean review text.
6. Remove unusable and duplicate rows.
7. Create a common `review_text` field.
8. Create simple sentiment labels from numeric ratings.
9. Save cleaned training and external-evaluation files.

> The Bright Data sample is intentionally kept separate from the Kaggle training data to reduce leakage and provide an out-of-time external evaluation set.



## 1. Setup

Download the Kaggle CSV and place it inside:

`data/raw/`

The notebook identifies the Kaggle file automatically by checking for expected columns, so the exact CSV filename is not important.

For Bright Data, the notebook first looks for:

`data/raw/glassdoor-companies-reviews.csv`

If it is not present, it attempts to read the public raw GitHub CSV directly.


In [21]:

from pathlib import Path
import html
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents = True, exist_ok = True)
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

BRIGHT_LOCAL_PATH = RAW_DIR / "glassdoor-companies-reviews.csv"
BRIGHT_RAW_URL = (
    "https://raw.githubusercontent.com/"
    "luminati-io/Glassdoor-dataset-samples/main/"
    "glassdoor-companies-reviews.csv"
)

print("Raw data folder:", RAW_DIR.resolve())
print("Processed data folder:", PROCESSED_DIR.resolve())


Raw data folder: /Users/shujiazhang/Desktop/2026Fall/APAN5560/Project/data/raw
Processed data folder: /Users/shujiazhang/Desktop/2026Fall/APAN5560/Project/data/processed



## 2. Load the raw datasets

### Kaggle expected fields

The Kaggle source is expected to contain fields such as:

`firm`, `date_review`, `job_title`, `current`, `location`, `overall_rating`,
`work_life_balance`, `culture_values`, `diversity_inclusion`, `career_opp`,
`comp_benefits`, `senior_mgmt`, `recommend`, `ceo_approv`, `outlook`,
`headline`, `pros`, and `cons`.

### Bright Data / GitHub field mapping

The Bright Data sample uses different names for equivalent concepts:

- `company_name` → `firm`
- `rating_date` → `date_review`
- `review_pros` → `pros`
- `review_cons` → `cons`
- `rating_work_life` → `work_life_balance`
- `rating_compensation_benefits` → `comp_benefits`
- `rating_culture_values` → `culture_values`
- `rating_career_opportunities` → `career_opp`
- `rating_senior_leadership` → `senior_mgmt`


In [22]:

KAGGLE_REQUIRED = {
    "firm",
    "date_review",
    "overall_rating",
    "pros",
    "cons",
}

def find_kaggle_csv(search_dirs = (RAW_DIR, Path("."))):
    candidates = []
    for folder in search_dirs:
        if folder.exists():
            candidates.extend(folder.glob("*.csv"))

    for path in candidates:
        if path.name == BRIGHT_LOCAL_PATH.name:
            continue

        try:
            sample = pd.read_csv(path, nrows = 5, low_memory = False)
        except Exception:
            continue

        if KAGGLE_REQUIRED.issubset(sample.columns):
            return path

    raise FileNotFoundError(
        "Kaggle CSV not found. Download the Kaggle Glassdoor Job Reviews dataset "
        "and place its CSV file inside data/raw/."
    )

kaggle_path = find_kaggle_csv()
kaggle_raw = pd.read_csv(kaggle_path, low_memory = False)

print("Kaggle file:", kaggle_path)
print("Kaggle shape:", kaggle_raw.shape)
display(kaggle_raw.head(3))


Kaggle file: data/raw/glassdoor_reviews.csv
Kaggle shape: (838566, 18)


,firm,date_review,job_title,current,location,overall_rating,work_life_balance,culture_values,diversity_inclusion,career_opp,comp_benefits,senior_mgmt,recommend,ceo_approv,outlook,headline,pros,cons
0,AFH-Wealth-Management,2015-04-05,,Current Employee,NaN,2,4.0,3.0,NaN,2.0,3.0,3.0,x,o,r,"Young colleagues, poor micro management",Very friendly and welcoming to new staff. Easy going ethic.,"Poor salaries, poor training and communication."
1,AFH-Wealth-Management,2015-12-11,Office Administrator,"Current Employee, more than 1 year","Bromsgrove, England, England",2,3.0,1.0,NaN,2.0,1.0,4.0,x,o,r,"Excellent staff, poor salary","Friendly, helpful and hard-working colleagues","Poor salary which doesn't improve much with progression, no incentive to work harder, high turnover of staff, poor s..."
2,AFH-Wealth-Management,2016-01-28,Office Administrator,"Current Employee, less than 1 year","Bromsgrove, England, England",1,1.0,1.0,NaN,1.0,1.0,1.0,x,o,x,"Low salary, bad micromanagement",Easy to get the job even without experience in finance,"Very low salary, poor working conditions, very little training provided but high expectations. Micro management are ..."


In [23]:

def load_bright_data():
    if BRIGHT_LOCAL_PATH.exists():
        print("Loading local Bright Data file:", BRIGHT_LOCAL_PATH)
        return pd.read_csv(BRIGHT_LOCAL_PATH, low_memory = False)

    print("Local Bright Data file not found; reading public GitHub CSV.")
    try:
        return pd.read_csv(BRIGHT_RAW_URL, low_memory = False)
    except Exception as exc:
        raise RuntimeError(
            "Could not load the Bright Data CSV from GitHub. "
            "Download glassdoor-companies-reviews.csv from the repository "
            "and place it inside data/raw/."
        ) from exc

bright_raw = load_bright_data()

print("Bright Data shape:", bright_raw.shape)
display(bright_raw.head(3))


Local Bright Data file not found; reading public GitHub CSV.
Bright Data shape: (1000, 36)


,timestamp,overview_id,review_id,review_url,rating_date,count_helpful,count_unhelpful,employee_job_end_year,employee_length,employee_responses,employee_status,employee_type,flag_covid,flag_featured,flags_business_outlook,flags_ceo_approval,flags_recommend_frend,rating_culture_values,rating_diversity_inclusion,rating_overall,rating_work_life,summary,company_name,review_advice,url,employee_location,employee_job_title,advice_to_management,overview_url,original_url,review_pros,review_cons,rating_compensation_benefits,rating_senior_leadership,rating_career_opportunities,glassdoor_employer_id
0,2024-04-02 00:00:00.000,925157,28951745,https://www.glassdoor.co.uk/Reviews/Employee-Review-Elevance-Health-RVW28951745.htm,"""2019-08-30T18:24:32.680Z""",10.0,NaN,2019.0,20.0,NaN,REGULAR,"Former employee, more than 10 years",NaN,NaN,NEGATIVE,DISAPPROVE,NEGATIVE,1,NaN,2,1,IT is Out of Control,Elevance Health,NaN,https://www.glassdoor.co.uk/Reviews/Elevance-Health-Reviews-E925157_P392.htm?sort.sortType=RD&sort.ascending=false&f...,NaN,Manager,Humble yourselves. Remember when you were coming up in your career. Respect your associates and appreciate all tha...,"https://www.glassdoor.co.uk/Overview/Working-at-Elevance-Health-EI_IE925157.11,26.htm",https://www.glassdoor.co.uk/Reviews/Elevance-Health-Reviews-E925157.htm?filter.iso3Language=spa&filter.employmentSta...,"Salary was good for the position, but only because they never promoted anyone.",Relentless pressure from the top. Constantly changing direction. Senior management is easily sold on every shiny n...,2,1,1.0,925157.0
1,2024-04-02 00:00:00.000,100231,59978917,https://www.glassdoor.co.uk/Reviews/Employee-Review-HDFC-Bank-RVW59978917.htm,"""2022-02-19T21:38:00.363Z""",2.0,NaN,NaN,6.0,NaN,REGULAR,"Current employee, more than 5 years",NaN,NaN,POSITIVE,APPROVE,POSITIVE,3,5.0,5,2,Great Company to work with,HDFC Bank,NaN,https://www.glassdoor.co.uk/Reviews/HDFC-Bank-Reviews-E100231_P296.htm?sort.sortType=RD&sort.ascending=false&filter....,New Delhi,Assistant Vice President,"Bring this great Company at par with similar great companies in India, eg, Infosys, TATA, etc, in terms of work cult...","https://www.glassdoor.co.uk/Overview/Working-at-HDFC-Bank-EI_IE100231.11,20.htm",https://www.glassdoor.co.uk/Reviews/HDFC-Bank-Reviews-E100231.htm?filter.iso3Language=spa&filter.employmentStatus=IN...,1. Great brand name\r\n2. Top Management are consistent on bank's goals and strategies\r\n3. Constantly growing company,1. Work hours do not adhere to internationally accepted standards\r\n2. Supervisory culture still embedded in 20th c...,3,5,5.0,100231.0
2,2024-04-02 00:00:00.000,9079,42749601,https://www.glassdoor.co.uk/Reviews/Employee-Review-Google-RVW42749601.htm,"""2021-02-19T02:53:22.840Z""",NaN,NaN,2021.0,2.0,NaN,REGULAR,"Former employee, more than 1 year",NaN,NaN,POSITIVE,APPROVE,POSITIVE,5,5.0,4,5,Great,Google,NaN,https://www.glassdoor.co.uk/Reviews/Google-Reviews-E9079_P3013.htm?sort.sortType=RD&sort.ascending=false&filter.iso3...,Chennai,Software Developer,Good,"https://www.glassdoor.co.uk/Overview/Working-at-Google-EI_IE9079.11,17.htm",https://www.glassdoor.co.uk/Reviews/Google-Reviews-E9079.htm?filter.iso3Language=spa&filter.employmentStatus=INTERN&...,Good and best and bestest best,Not bad and not bad and not bad,5,5,3.0,9079.0



## 3. Inspect the raw schemas

Makes source changes visible before cleaning. Optional missing columns will be added as `NaN` later so both datasets can still be transformed into one common schema.


In [24]:

print("KAGGLE COLUMNS")
print(kaggle_raw.columns.tolist())

print("\nBRIGHT DATA COLUMNS")
print(bright_raw.columns.tolist())


KAGGLE COLUMNS
['firm', 'date_review', 'job_title', 'current', 'location', 'overall_rating', 'work_life_balance', 'culture_values', 'diversity_inclusion', 'career_opp', 'comp_benefits', 'senior_mgmt', 'recommend', 'ceo_approv', 'outlook', 'headline', 'pros', 'cons']

BRIGHT DATA COLUMNS
['timestamp', 'overview_id', 'review_id', 'review_url', 'rating_date', 'count_helpful', 'count_unhelpful', 'employee_job_end_year', 'employee_length', 'employee_responses', 'employee_status', 'employee_type', 'flag_covid', 'flag_featured', 'flags_business_outlook', 'flags_ceo_approval', 'flags_recommend_frend', 'rating_culture_values', 'rating_diversity_inclusion', 'rating_overall', 'rating_work_life', 'summary', 'company_name', 'review_advice', 'url', 'employee_location', 'employee_job_title', 'advice_to_management', 'overview_url', 'original_url', 'review_pros', 'review_cons', 'rating_compensation_benefits', 'rating_senior_leadership', 'rating_career_opportunities', 'glassdoor_employer_id']



## 4. Define the common schema

The common schema keeps fields useful for:

- text classification;
- sentiment prediction;
- aspect-level prediction;
- external evaluation;
- later RAG retrieval.

The Bright Data field `timestamp` is **not** used as the review date. We use `rating_date`, which represents when the review was submitted.


In [25]:

COMMON_COLUMNS = [
    "firm",
    "date_review",
    "job_title",
    "current",
    "location",
    "overall_rating",
    "work_life_balance",
    "culture_values",
    "diversity_inclusion",
    "career_opp",
    "comp_benefits",
    "senior_mgmt",
    "recommend",
    "ceo_approv",
    "outlook",
    "headline",
    "pros",
    "cons",
    "review_id",
    "review_url",
    "source",
]

ASPECT_COLUMNS = [
    "work_life_balance",
    "culture_values",
    "diversity_inclusion",
    "career_opp",
    "comp_benefits",
    "senior_mgmt",
]

BRIGHT_RENAME = {
    "company_name": "firm",
    "rating_date": "date_review",
    "employee_job_title": "job_title",
    "employee_type": "current",
    "employee_location": "location",
    "rating_overall": "overall_rating",
    "rating_work_life": "work_life_balance",
    "rating_culture_values": "culture_values",
    "rating_diversity_inclusion": "diversity_inclusion",
    "rating_career_opportunities": "career_opp",
    "rating_compensation_benefits": "comp_benefits",
    "rating_senior_leadership": "senior_mgmt",
    "flags_recommend_frend": "recommend",
    "flags_ceo_approval": "ceo_approv",
    "flags_business_outlook": "outlook",
    "summary": "headline",
    "review_pros": "pros",
    "review_cons": "cons",
}



## 5. Cleaning helpers

For transformer / LLM training, text is cleaned **lightly**.

We:
- decode HTML entities such as `&amp;`;
- remove HTML tags;
- remove embedded URLs;
- normalize whitespace;
- preserve normal casing, punctuation, and wording.

We do **not** stem, lemmatize, remove stopwords, or strip punctuation because modern language models benefit from retaining the original language structure.


In [26]:

NULL_STRINGS = {
    "",
    "nan",
    "none",
    "null",
    "n/a",
    "na",
    "unknown",
}

def clean_text(value):
    if pd.isna(value):
        return np.nan

    text = html.unescape(str(value))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    if text.lower() in NULL_STRINGS:
        return np.nan

    return text if text else np.nan


def clean_rating(series):
    # Convert ratings to numeric and keep only the valid 1-5 scale.
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.where(numeric.between(1, 5))


def normalize_token(value):
    if pd.isna(value):
        return np.nan

    token = str(value).strip().lower()
    token = re.sub(r"[\s\-]+", "_", token)

    if token in NULL_STRINGS:
        return np.nan

    return token


def normalize_recommend(value):
    token = normalize_token(value)
    if pd.isna(token):
        return np.nan

    mapping = {
        "positive": "positive",
        "negative": "negative",
        "neutral": "neutral",
        "no_opinion": "no_opinion",
        "v": "positive",
        "r": "neutral",
        "x": "negative",
        "o": "no_opinion",
        "recommend": "positive",
        "recommended": "positive",
        "not_recommend": "negative",
        "not_recommended": "negative",
        "true": "positive",
        "false": "negative",
        "1": "positive",
        "0": "negative",
    }
    return mapping.get(token, token)


def normalize_ceo(value):
    token = normalize_token(value)
    if pd.isna(token):
        return np.nan

    mapping = {
        "approve": "positive",
        "approved": "positive",
        "positive": "positive",
        "disapprove": "negative",
        "disapproved": "negative",
        "negative": "negative",
        "neutral": "neutral",
        "no_opinion": "no_opinion",
        "v": "positive",
        "r": "neutral",
        "x": "negative",
        "o": "no_opinion",
    }
    return mapping.get(token, token)


def normalize_outlook(value):
    token = normalize_token(value)
    if pd.isna(token):
        return np.nan

    mapping = {
        "positive": "positive",
        "negative": "negative",
        "neutral": "neutral",
        "no_opinion": "no_opinion",
        "v": "positive",
        "r": "neutral",
        "x": "negative",
        "o": "no_opinion",
    }
    return mapping.get(token, token)


def rating_to_sentiment(value):
    # Derived target: 1-2 negative, 3 neutral, 4-5 positive.
    if pd.isna(value):
        return np.nan
    if value <= 2:
        return "negative"
    if value == 3:
        return "neutral"
    return "positive"


def ensure_columns(df, columns):
    out = df.copy()
    for column in columns:
        if column not in out.columns:
            out[column] = np.nan
    return out



## 6. Clean the Kaggle dataset

The Kaggle source remains the **primary training source**.

Cleaning steps:

- parse review dates;
- keep valid ratings from 1 to 5;
- normalize categorical variables;
- lightly clean text;
- combine title, pros, and cons into one `review_text`;
- remove unusable short reviews;
- remove duplicate reviews.


In [27]:

def clean_common_fields(df, source_name):
    out = ensure_columns(df, COMMON_COLUMNS).copy()
    out["source"] = source_name

    out["date_review"] = (out["date_review"].astype(str).str.replace('"', '', regex = False).str.strip())
    out["date_review"] = pd.to_datetime(out["date_review"], errors = "coerce", utc = True)

    rating_columns = ["overall_rating"] + ASPECT_COLUMNS
    for column in rating_columns:
        out[column] = clean_rating(out[column])

    text_columns = [
        "firm",
        "job_title",
        "current",
        "location",
        "headline",
        "pros",
        "cons",
        "review_id",
        "review_url",
    ]

    for column in text_columns:
        out[column] = out[column].map(clean_text)

    out["recommend"] = out["recommend"].map(normalize_recommend)
    out["ceo_approv"] = out["ceo_approv"].map(normalize_ceo)
    out["outlook"] = out["outlook"].map(normalize_outlook)

    def combine_review_text(row):
        parts = []

        if pd.notna(row["headline"]):
            parts.append(f"TITLE: {row['headline']}")
        if pd.notna(row["pros"]):
            parts.append(f"PROS: {row['pros']}")
        if pd.notna(row["cons"]):
            parts.append(f"CONS: {row['cons']}")

        text = " ".join(parts).strip()
        return text if text else np.nan

    out["review_text"] = out.apply(combine_review_text, axis = 1)
    out["text_length"] = out["review_text"].fillna("").str.len()

    out["sentiment_label"] = out["overall_rating"].map(
        rating_to_sentiment
    )

    out = out[out["review_text"].notna()].copy()
    out = out[out["text_length"] >= 20].copy()

    return out


kaggle_clean = clean_common_fields(
    kaggle_raw,
    source_name = "kaggle_glassdoor",
)

before = len(kaggle_clean)

kaggle_clean = kaggle_clean.drop_duplicates(
    subset = ["firm", "date_review", "review_text"],
    keep = "first",
).reset_index(drop = True)

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", len(kaggle_clean))
print(
    "Date range:",
    kaggle_clean["date_review"].min(),
    "to",
    kaggle_clean["date_review"].max(),
)

display(kaggle_clean.head(3))


Rows before duplicate removal: 838566
Rows after duplicate removal: 803631
Date range: 2008-01-31 00:00:00+00:00 to 2021-06-08 00:00:00+00:00


,firm,date_review,job_title,current,location,overall_rating,work_life_balance,culture_values,diversity_inclusion,career_opp,comp_benefits,senior_mgmt,recommend,ceo_approv,outlook,headline,pros,cons,review_id,review_url,source,review_text,text_length,sentiment_label
0,AFH-Wealth-Management,2015-04-05 00:00:00+00:00,NaN,Current Employee,NaN,2,4.0,3.0,NaN,2.0,3.0,3.0,negative,no_opinion,neutral,"Young colleagues, poor micro management",Very friendly and welcoming to new staff. Easy going ethic.,"Poor salaries, poor training and communication.",NaN,NaN,kaggle_glassdoor,"TITLE: Young colleagues, poor micro management PROS: Very friendly and welcoming to new staff. Easy going ethic. CON...",166,negative
1,AFH-Wealth-Management,2015-12-11 00:00:00+00:00,Office Administrator,"Current Employee, more than 1 year","Bromsgrove, England, England",2,3.0,1.0,NaN,2.0,1.0,4.0,negative,no_opinion,neutral,"Excellent staff, poor salary","Friendly, helpful and hard-working colleagues","Poor salary which doesn't improve much with progression, no incentive to work harder, high turnover of staff, poor s...",NaN,NaN,kaggle_glassdoor,"TITLE: Excellent staff, poor salary PROS: Friendly, helpful and hard-working colleagues CONS: Poor salary which does...",216,negative
2,AFH-Wealth-Management,2016-01-28 00:00:00+00:00,Office Administrator,"Current Employee, less than 1 year","Bromsgrove, England, England",1,1.0,1.0,NaN,1.0,1.0,1.0,negative,no_opinion,negative,"Low salary, bad micromanagement",Easy to get the job even without experience in finance,"Very low salary, poor working conditions, very little training provided but high expectations. Micro management are ...",NaN,NaN,kaggle_glassdoor,"TITLE: Low salary, bad micromanagement PROS: Easy to get the job even without experience in finance CONS: Very low s...",330,negative



## 7. Clean and standardize Bright Data

The Bright Data source is first renamed into the common Kaggle-style schema.

The same cleaning rules are then applied so both datasets are processed consistently.


In [28]:

bright_standardized = bright_raw.rename(
    columns = BRIGHT_RENAME
).copy()

bright_clean = clean_common_fields(
    bright_standardized,
    source_name = "brightdata_glassdoor",
)

before = len(bright_clean)

bright_clean = bright_clean.drop_duplicates(
    subset = ["firm", "date_review", "review_text"],
    keep = "first",
).reset_index(drop = True)

print("Rows before duplicate removal:", before)
print("Rows after duplicate removal:", len(bright_clean))
print(
    "Full Bright Data review-date range:",
    bright_clean["date_review"].min(),
    "to",
    bright_clean["date_review"].max(),
)

display(bright_clean.head(3))


Rows before duplicate removal: 1000
Rows after duplicate removal: 1000
Full Bright Data review-date range: 2012-07-09 21:07:34.857000+00:00 to 2024-03-24 13:14:52.137000+00:00


,timestamp,overview_id,review_id,review_url,date_review,count_helpful,count_unhelpful,employee_job_end_year,employee_length,employee_responses,employee_status,current,flag_covid,flag_featured,outlook,ceo_approv,recommend,culture_values,diversity_inclusion,overall_rating,work_life_balance,headline,firm,review_advice,url,location,job_title,advice_to_management,overview_url,original_url,pros,cons,comp_benefits,senior_mgmt,career_opp,glassdoor_employer_id,source,review_text,text_length,sentiment_label
0,2024-04-02 00:00:00.000,925157,28951745,NaN,2019-08-30 18:24:32.680000+00:00,10.0,NaN,2019.0,20.0,NaN,REGULAR,"Former employee, more than 10 years",NaN,NaN,negative,negative,negative,1,NaN,2,1,IT is Out of Control,Elevance Health,NaN,https://www.glassdoor.co.uk/Reviews/Elevance-Health-Reviews-E925157_P392.htm?sort.sortType=RD&sort.ascending=false&f...,NaN,Manager,Humble yourselves. Remember when you were coming up in your career. Respect your associates and appreciate all tha...,"https://www.glassdoor.co.uk/Overview/Working-at-Elevance-Health-EI_IE925157.11,26.htm",https://www.glassdoor.co.uk/Reviews/Elevance-Health-Reviews-E925157.htm?filter.iso3Language=spa&filter.employmentSta...,"Salary was good for the position, but only because they never promoted anyone.",Relentless pressure from the top. Constantly changing direction. Senior management is easily sold on every shiny new...,2,1,1.0,925157.0,brightdata_glassdoor,"TITLE: IT is Out of Control PROS: Salary was good for the position, but only because they never promoted anyone. CON...",291,negative
1,2024-04-02 00:00:00.000,100231,59978917,NaN,2022-02-19 21:38:00.363000+00:00,2.0,NaN,NaN,6.0,NaN,REGULAR,"Current employee, more than 5 years",NaN,NaN,positive,positive,positive,3,5.0,5,2,Great Company to work with,HDFC Bank,NaN,https://www.glassdoor.co.uk/Reviews/HDFC-Bank-Reviews-E100231_P296.htm?sort.sortType=RD&sort.ascending=false&filter....,New Delhi,Assistant Vice President,"Bring this great Company at par with similar great companies in India, eg, Infosys, TATA, etc, in terms of work cult...","https://www.glassdoor.co.uk/Overview/Working-at-HDFC-Bank-EI_IE100231.11,20.htm",https://www.glassdoor.co.uk/Reviews/HDFC-Bank-Reviews-E100231.htm?filter.iso3Language=spa&filter.employmentStatus=IN...,1. Great brand name 2. Top Management are consistent on bank's goals and strategies 3. Constantly growing company,1. Work hours do not adhere to internationally accepted standards 2. Supervisory culture still embedded in 20th cent...,3,5,5.0,100231.0,brightdata_glassdoor,TITLE: Great Company to work with PROS: 1. Great brand name 2. Top Management are consistent on bank's goals and str...,378,positive
2,2024-04-02 00:00:00.000,9079,42749601,NaN,2021-02-19 02:53:22.840000+00:00,NaN,NaN,2021.0,2.0,NaN,REGULAR,"Former employee, more than 1 year",NaN,NaN,positive,positive,positive,5,5.0,4,5,Great,Google,NaN,https://www.glassdoor.co.uk/Reviews/Google-Reviews-E9079_P3013.htm?sort.sortType=RD&sort.ascending=false&filter.iso3...,Chennai,Software Developer,Good,"https://www.glassdoor.co.uk/Overview/Working-at-Google-EI_IE9079.11,17.htm",https://www.glassdoor.co.uk/Reviews/Google-Reviews-E9079.htm?filter.iso3Language=spa&filter.employmentStatus=INTERN&...,Good and best and bestest best,Not bad and not bad and not bad,5,5,3.0,9079.0,brightdata_glassdoor,TITLE: Great PROS: Good and best and bestest best CONS: Not bad and not bad and not bad,87,positive



## 8. Keep recent Bright Data reviews

To make the supplementary source useful as an out-of-time dataset, keep only reviews dated **January 1, 2022 or later**.

This reduces temporal overlap with the Kaggle source and better represents recent employee feedback.


In [29]:

RECENT_START = pd.Timestamp("2022-01-01", tz="UTC")

bright_recent = bright_clean[
    bright_clean["date_review"].notna()
    & (bright_clean["date_review"] >= RECENT_START)
].copy()

bright_recent = bright_recent.reset_index(drop = True)

print("Bright Data rows before recent-date filter:", len(bright_clean))
print("Bright Data rows from 2022 onward:", len(bright_recent))

if len(bright_recent) > 0:
    print(
        "Recent review-date range:",
        bright_recent["date_review"].min(),
        "to",
        bright_recent["date_review"].max(),
    )

display(bright_recent.head(3))


Bright Data rows before recent-date filter: 1000
Bright Data rows from 2022 onward: 431
Recent review-date range: 2022-01-02 04:42:41.397000+00:00 to 2024-03-24 13:14:52.137000+00:00


,timestamp,overview_id,review_id,review_url,date_review,count_helpful,count_unhelpful,employee_job_end_year,employee_length,employee_responses,employee_status,current,flag_covid,flag_featured,outlook,ceo_approv,recommend,culture_values,diversity_inclusion,overall_rating,work_life_balance,headline,firm,review_advice,url,location,job_title,advice_to_management,overview_url,original_url,pros,cons,comp_benefits,senior_mgmt,career_opp,glassdoor_employer_id,source,review_text,text_length,sentiment_label
0,2024-04-02 00:00:00.000,100231,59978917,NaN,2022-02-19 21:38:00.363000+00:00,2.0,NaN,NaN,6.0,NaN,REGULAR,"Current employee, more than 5 years",NaN,NaN,positive,positive,positive,3,5.0,5,2,Great Company to work with,HDFC Bank,NaN,https://www.glassdoor.co.uk/Reviews/HDFC-Bank-Reviews-E100231_P296.htm?sort.sortType=RD&sort.ascending=false&filter....,New Delhi,Assistant Vice President,"Bring this great Company at par with similar great companies in India, eg, Infosys, TATA, etc, in terms of work cult...","https://www.glassdoor.co.uk/Overview/Working-at-HDFC-Bank-EI_IE100231.11,20.htm",https://www.glassdoor.co.uk/Reviews/HDFC-Bank-Reviews-E100231.htm?filter.iso3Language=spa&filter.employmentStatus=IN...,1. Great brand name 2. Top Management are consistent on bank's goals and strategies 3. Constantly growing company,1. Work hours do not adhere to internationally accepted standards 2. Supervisory culture still embedded in 20th cent...,3,5,5.0,100231.0,brightdata_glassdoor,TITLE: Great Company to work with PROS: 1. Great brand name 2. Top Management are consistent on bank's goals and str...,378,positive
1,2023-07-28 00:00:00.000,58316203,58316203,NaN,2022-01-21 00:00:00+00:00,1.0,NaN,2021.0,2.0,NaN,NaN,Former Employee,NaN,NaN,positive,positive,positive,5,5.0,5,5,Commercial Product Internship,Florida Blue,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Very inclusive and helpful environment which made succeeding in the role very achievable.,Nothing sticks out to me that I would change.,5,5,NaN,NaN,brightdata_glassdoor,TITLE: Commercial Product Internship PROS: Very inclusive and helpful environment which made succeeding in the role ...,184,positive
2,2023-08-03 00:00:00.000,42257812,77730254,NaN,2023-06-26 00:00:00+00:00,3.0,NaN,2023.0,2.0,NaN,NaN,Former Employee,NaN,NaN,negative,negative,negative,1,1.0,1,1,Regrettable Experience - A Workplace To Be Wary Of,Chargebee,NaN,NaN,NaN,NaN,NaN,NaN,NaN,- Eye-opening: Teaches you exactly how a company should not be run. - Pays the bills: The compensation is tolerable....,- Communication breakdown: There is an endemic lack of communication that exacerbates every problem within the compa...,2,1,NaN,NaN,brightdata_glassdoor,TITLE: Regrettable Experience - A Workplace To Be Wary Of PROS: - Eye-opening: Teaches you exactly how a company sho...,754,negative



## 9. Cross-source duplicate check

Even after filtering by date, check whether normalized review text appears in both sources.

Any matching row is removed from the Bright Data external set to reduce leakage.


In [30]:

def normalize_for_duplicate_check(text):
    if pd.isna(text):
        return np.nan

    text = str(text).lower()
    text = re.sub(r"\W+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


kaggle_text_keys = set(
    kaggle_clean["review_text"]
    .map(normalize_for_duplicate_check)
    .dropna()
)

bright_recent["duplicate_key"] = bright_recent["review_text"].map(
    normalize_for_duplicate_check
)

cross_source_duplicate_mask = bright_recent["duplicate_key"].isin(
    kaggle_text_keys
)

print(
    "Cross-source duplicate rows found:",
    int(cross_source_duplicate_mask.sum()),
)

bright_recent = (
    bright_recent.loc[~cross_source_duplicate_mask]
    .drop(columns="duplicate_key")
    .reset_index(drop=True)
)


Cross-source duplicate rows found: 0



## 10. Prepare training-ready table functions

This section defines reusable functions and creates full cleaned tables for exploration. The actual Kaggle train/validation/test datasets are created later at the **review level** so that one review cannot leak across splits.


### Overall sentiment

The notebook creates a simple derived sentiment label from `overall_rating`:

- 1–2 → `negative`
- 3 → `neutral`
- 4–5 → `positive`

### Aspect-level data

For every non-missing aspect rating, one row is created containing:

- `review_text`
- `aspect`
- `aspect_rating`
- `aspect_sentiment`
- `task_text`

This supports aspect-sentiment training.

> Important: these aspect ratings are **not true theme-presence labels**. A rating tells us how the employee scored a dimension, but it does not prove that the review text explicitly discusses that theme.


In [31]:

def make_sentiment_table(df):
    columns = [
        "source",
        "firm",
        "date_review",
        "job_title",
        "location",
        "review_text",
        "overall_rating",
        "sentiment_label",
    ]

    out = df[columns].copy()
    out = out.dropna(
        subset=["review_text", "sentiment_label"]
    )

    return out.reset_index(drop=True)


def make_aspect_table(df):
    records = []

    meta_columns = [
        "source",
        "firm",
        "date_review",
        "job_title",
        "location",
        "review_text",
    ]

    for aspect in ASPECT_COLUMNS:
        subset = df.loc[
            df[aspect].notna(),
            meta_columns + [aspect],
        ].copy()

        subset = subset.rename(
            columns={aspect: "aspect_rating"}
        )

        subset["aspect"] = aspect

        subset["aspect_sentiment"] = (
            subset["aspect_rating"]
            .map(rating_to_sentiment)
        )

        subset["task_text"] = (
            "ASPECT: "
            + subset["aspect"]
            + " REVIEW: "
            + subset["review_text"]
        )

        records.append(subset)

    if not records:
        return pd.DataFrame()

    out = pd.concat(records, ignore_index=True)

    return out[
        meta_columns
        + [
            "aspect",
            "aspect_rating",
            "aspect_sentiment",
            "task_text",
        ]
    ]


kaggle_sentiment_all = make_sentiment_table(
    kaggle_clean
)

kaggle_aspect_all = make_aspect_table(
    kaggle_clean
)

bright_sentiment_eval = make_sentiment_table(
    bright_recent
)

bright_aspect_eval = make_aspect_table(
    bright_recent
)

print(
    "Kaggle overall-sentiment rows (all cleaned data):",
    len(kaggle_sentiment_all),
)

print(
    "Kaggle aspect rows (all cleaned data):",
    len(kaggle_aspect_all),
)

print(
    "Bright recent overall-sentiment rows:",
    len(bright_sentiment_eval),
)

print(
    "Bright recent aspect rows:",
    len(bright_aspect_eval),
)


Kaggle overall-sentiment rows (all cleaned data): 803631
Kaggle aspect rows (all cleaned data): 3386468
Bright recent overall-sentiment rows: 431
Bright recent aspect rows: 2298



## 11. Explore the cleaned datasets

Before splitting the Kaggle data, we inspect the two cleaned sources to understand their size, date coverage, missingness, label balance, and text length.

The goal is not a full EDA. These checks focus on variables that matter for model training and external evaluation:

- number of usable reviews;
- date range;
- overall rating and sentiment distribution;
- missingness in the six aspect ratings;
- review-text length;
- review counts by year.

The Bright Data dataset remains separate and represents the newer **2022–2024 external evaluation source**.


In [32]:

def basic_dataset_summary(name, df):
    return {
        "dataset": name,
        "rows": len(df),
        "firms": df["firm"].nunique(dropna=True),
        "start_date": df["date_review"].min(),
        "end_date": df["date_review"].max(),
        "avg_overall_rating": df["overall_rating"].mean(),
        "median_text_length": df["text_length"].median(),
        "avg_text_length": df["text_length"].mean(),
    }


dataset_summary = pd.DataFrame([
    basic_dataset_summary("Kaggle cleaned", kaggle_clean),
    basic_dataset_summary("Bright Data 2022+", bright_recent),
])

dataset_summary["avg_overall_rating"] = (
    dataset_summary["avg_overall_rating"].round(2)
)
dataset_summary["median_text_length"] = (
    dataset_summary["median_text_length"].round(1)
)
dataset_summary["avg_text_length"] = (
    dataset_summary["avg_text_length"].round(1)
)

display(dataset_summary)


,dataset,rows,firms,start_date,end_date,avg_overall_rating,median_text_length,avg_text_length
0,Kaggle cleaned,803631,428,2008-01-31 00:00:00+00:00,2021-06-08 00:00:00+00:00,3.66,164.0,257.7
1,Bright Data 2022+,431,338,2022-01-02 04:42:41.397000+00:00,2024-03-24 13:14:52.137000+00:00,2.36,286.0,519.0


In [33]:

# Compare missingness in the rating variables.

rating_columns = ["overall_rating"] + ASPECT_COLUMNS

missingness_comparison = pd.DataFrame({
    "Kaggle_missing_%": (
        kaggle_clean[rating_columns]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    ),
    "Bright_2022plus_missing_%": (
        bright_recent[rating_columns]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    ),
})

display(missingness_comparison)


,Kaggle_missing_%,Bright_2022plus_missing_%
overall_rating,0.00,0.00
work_life_balance,17.91,0.00
culture_values,22.79,0.00
diversity_inclusion,83.75,0.00
career_opp,17.62,66.82
comp_benefits,17.93,0.00
senior_mgmt,18.62,0.00


In [34]:

# Compare overall sentiment distributions.

kaggle_sentiment_distribution = (
    kaggle_sentiment_all["sentiment_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("Kaggle_%")
)

bright_sentiment_distribution = (
    bright_sentiment_eval["sentiment_label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("Bright_2022plus_%")
)

sentiment_comparison = pd.concat(
    [
        kaggle_sentiment_distribution,
        bright_sentiment_distribution,
    ],
    axis=1,
).fillna(0)

display(sentiment_comparison)


,Kaggle_%,Bright_2022plus_%
sentiment_label,,
positive,60.94,24.59
neutral,23.07,16.24
negative,15.99,59.16


In [35]:

# Review counts by year.

kaggle_reviews_by_year = (
    kaggle_clean
    .assign(year=kaggle_clean["date_review"].dt.year)
    .groupby("year")
    .size()
    .rename("Kaggle_reviews")
)

bright_reviews_by_year = (
    bright_recent
    .assign(year=bright_recent["date_review"].dt.year)
    .groupby("year")
    .size()
    .rename("Bright_reviews")
)

reviews_by_year = pd.concat(
    [kaggle_reviews_by_year, bright_reviews_by_year],
    axis=1,
).fillna(0).astype(int)

display(reviews_by_year)


,Kaggle_reviews,Bright_reviews
year,,
2008,5338,0
2009,5645,0
2010,9519,0
2011,10314,0
2012,19462,0
2013,23538,0
2014,37834,0
2015,66285,0
2016,80833,0


In [36]:

# Review-text length summary.

text_length_summary = pd.DataFrame({
    "Kaggle": kaggle_clean["text_length"].describe(),
    "Bright_2022plus": bright_recent["text_length"].describe(),
}).round(2)

display(text_length_summary)


,Kaggle,Bright_2022plus
count,803631.00,431.00
mean,257.66,519.00
std,296.16,611.82
min,47.00,77.00
25%,120.00,164.50
50%,164.00,286.00
75%,288.00,585.50
max,18623.00,4549.00



## 12. Define training, validation, and test datasets

We split the **Kaggle cleaned reviews** into:

- **80% training set** — used to fine-tune the model;
- **10% validation set** — used for model selection, hyperparameter decisions, and monitoring overfitting;
- **10% internal test set** — used once for final evaluation on unseen reviews from the same Kaggle source.

The **Bright Data 2022+ reviews are not included in this split**. They remain a completely separate **external test set** for checking whether a model trained on historical Kaggle reviews generalizes to newer employee feedback.

The split is performed at the **review level before creating aspect-level rows**. This prevents the same review from appearing in multiple splits through different aspect ratings.

We stratify by `sentiment_label` so that the overall positive / neutral / negative proportions remain similar across Kaggle train, validation, and internal test sets.


In [37]:

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Use reviews with an available overall sentiment label for supervised splitting.
kaggle_split_base = (
    kaggle_clean
    .dropna(subset=["review_text", "sentiment_label"])
    .copy()
)

# First split: 80% training, 20% temporary holdout.
kaggle_train_reviews, kaggle_holdout_reviews = train_test_split(
    kaggle_split_base,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=kaggle_split_base["sentiment_label"],
)

# Second split: divide the 20% holdout equally into
# 10% validation and 10% internal test.
kaggle_validation_reviews, kaggle_test_reviews = train_test_split(
    kaggle_holdout_reviews,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=kaggle_holdout_reviews["sentiment_label"],
)

# Reset indices after splitting.
kaggle_train_reviews = kaggle_train_reviews.reset_index(drop=True)
kaggle_validation_reviews = kaggle_validation_reviews.reset_index(drop=True)
kaggle_test_reviews = kaggle_test_reviews.reset_index(drop=True)

# Bright Data remains untouched as the external test set.
bright_external_test_reviews = bright_recent.copy().reset_index(drop=True)

print("Kaggle supervised review rows:", len(kaggle_split_base))
print("Training rows:", len(kaggle_train_reviews))
print("Validation rows:", len(kaggle_validation_reviews))
print("Internal test rows:", len(kaggle_test_reviews))
print("Bright external test rows:", len(bright_external_test_reviews))


Kaggle supervised review rows: 803631
Training rows: 642904
Validation rows: 80363
Internal test rows: 80364
Bright external test rows: 431


In [38]:

# Verify that the split proportions are approximately 80 / 10 / 10.

split_summary = pd.DataFrame({
    "split": [
        "Kaggle train",
        "Kaggle validation",
        "Kaggle internal test",
        "Bright external test",
    ],
    "rows": [
        len(kaggle_train_reviews),
        len(kaggle_validation_reviews),
        len(kaggle_test_reviews),
        len(bright_external_test_reviews),
    ],
})

kaggle_total = len(kaggle_split_base)

split_summary["percent_of_kaggle"] = [
    100 * len(kaggle_train_reviews) / kaggle_total,
    100 * len(kaggle_validation_reviews) / kaggle_total,
    100 * len(kaggle_test_reviews) / kaggle_total,
    np.nan,
]

split_summary["percent_of_kaggle"] = (
    split_summary["percent_of_kaggle"].round(2)
)

display(split_summary)


,split,rows,percent_of_kaggle
0,Kaggle train,642904,80.0
1,Kaggle validation,80363,10.0
2,Kaggle internal test,80364,10.0
3,Bright external test,431,NaN


In [39]:

# Check whether sentiment proportions remain similar across the three Kaggle splits.

def sentiment_percent(df):
    return (
        df["sentiment_label"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )


split_sentiment_balance = pd.DataFrame({
    "Train_%": sentiment_percent(kaggle_train_reviews),
    "Validation_%": sentiment_percent(kaggle_validation_reviews),
    "Internal_test_%": sentiment_percent(kaggle_test_reviews),
    "Bright_external_test_%": sentiment_percent(bright_external_test_reviews),
}).fillna(0)

display(split_sentiment_balance)


,Train_%,Validation_%,Internal_test_%,Bright_external_test_%
sentiment_label,,,,
negative,15.99,15.99,15.99,59.16
neutral,23.07,23.07,23.07,16.24
positive,60.94,60.94,60.94,24.59


In [40]:

# Create training-ready sentiment and aspect tables from each review-level split.

kaggle_sentiment_train = make_sentiment_table(
    kaggle_train_reviews
)

kaggle_sentiment_validation = make_sentiment_table(
    kaggle_validation_reviews
)

kaggle_sentiment_test = make_sentiment_table(
    kaggle_test_reviews
)

kaggle_aspect_train = make_aspect_table(
    kaggle_train_reviews
)

kaggle_aspect_validation = make_aspect_table(
    kaggle_validation_reviews
)

kaggle_aspect_test = make_aspect_table(
    kaggle_test_reviews
)

# Bright Data remains external only.
bright_sentiment_external_test = make_sentiment_table(
    bright_external_test_reviews
)

bright_aspect_external_test = make_aspect_table(
    bright_external_test_reviews
)

training_table_summary = pd.DataFrame({
    "table": [
        "Kaggle sentiment train",
        "Kaggle sentiment validation",
        "Kaggle sentiment internal test",
        "Kaggle aspect train",
        "Kaggle aspect validation",
        "Kaggle aspect internal test",
        "Bright sentiment external test",
        "Bright aspect external test",
    ],
    "rows": [
        len(kaggle_sentiment_train),
        len(kaggle_sentiment_validation),
        len(kaggle_sentiment_test),
        len(kaggle_aspect_train),
        len(kaggle_aspect_validation),
        len(kaggle_aspect_test),
        len(bright_sentiment_external_test),
        len(bright_aspect_external_test),
    ],
})

display(training_table_summary)


,table,rows
0,Kaggle sentiment train,642904
1,Kaggle sentiment validation,80363
2,Kaggle sentiment internal test,80364
3,Kaggle aspect train,2709126
4,Kaggle aspect validation,338400
5,Kaggle aspect internal test,338942
6,Bright sentiment external test,431
7,Bright aspect external test,2298



## 13. Quality checks

Before saving, inspect:

- dataset sizes;
- date ranges;
- missing review text;
- missingness of rating variables;
- rating ranges;
- sentiment distributions;
- number of labeled rows per aspect.


In [41]:

def quality_report(name, df):
    print("=" * 80)
    print(name)
    print("Rows:", len(df))

    if "date_review" in df.columns:
        print(
            "Date range:",
            df["date_review"].min(),
            "to",
            df["date_review"].max(),
        )

    if "review_text" in df.columns:
        print(
            "Missing review_text:",
            int(df["review_text"].isna().sum()),
        )

    rating_cols = [
        col
        for col in ["overall_rating"] + ASPECT_COLUMNS
        if col in df.columns
    ]

    if rating_cols:
        print("\nMissingness for ratings:")

        missing = (
            df[rating_cols]
            .isna()
            .mean()
            .mul(100)
            .round(2)
            .rename("missing_percent")
            .to_frame()
        )

        display(missing)

        print("Observed rating ranges:")

        for col in rating_cols:
            valid = df[col].dropna()

            if len(valid):
                print(
                    f"  {col}: "
                    f"{valid.min()} to {valid.max()}"
                )


quality_report(
    "Kaggle cleaned",
    kaggle_clean,
)

quality_report(
    "Bright Data recent cleaned",
    bright_recent,
)


Kaggle cleaned
Rows: 803631
Date range: 2008-01-31 00:00:00+00:00 to 2021-06-08 00:00:00+00:00
Missing review_text: 0

Missingness for ratings:


,missing_percent
overall_rating,0.00
work_life_balance,17.91
culture_values,22.79
diversity_inclusion,83.75
career_opp,17.62
comp_benefits,17.93
senior_mgmt,18.62


Observed rating ranges:
  overall_rating: 1 to 5
  work_life_balance: 1.0 to 5.0
  culture_values: 1.0 to 5.0
  diversity_inclusion: 1.0 to 5.0
  career_opp: 1.0 to 5.0
  comp_benefits: 1.0 to 5.0
  senior_mgmt: 1.0 to 5.0
Bright Data recent cleaned
Rows: 431
Date range: 2022-01-02 04:42:41.397000+00:00 to 2024-03-24 13:14:52.137000+00:00
Missing review_text: 0

Missingness for ratings:


,missing_percent
overall_rating,0.00
work_life_balance,0.00
culture_values,0.00
diversity_inclusion,0.00
career_opp,66.82
comp_benefits,0.00
senior_mgmt,0.00


Observed rating ranges:
  overall_rating: 1 to 5
  work_life_balance: 1 to 5
  culture_values: 1 to 5
  diversity_inclusion: 1.0 to 5.0
  career_opp: 1.0 to 5.0
  comp_benefits: 1 to 5
  senior_mgmt: 1 to 5


In [42]:

print("Kaggle training sentiment distribution")
display(
    kaggle_sentiment_train["sentiment_label"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nKaggle validation sentiment distribution")
display(
    kaggle_sentiment_validation["sentiment_label"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nKaggle internal-test sentiment distribution")
display(
    kaggle_sentiment_test["sentiment_label"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nBright external-test sentiment distribution")
display(
    bright_sentiment_external_test["sentiment_label"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nKaggle training aspect-label counts")
display(
    kaggle_aspect_train["aspect"]
    .value_counts()
    .to_frame("count")
)

print("\nBright external-test aspect-label counts")
display(
    bright_aspect_external_test["aspect"]
    .value_counts()
    .to_frame("count")
)


Kaggle training sentiment distribution


,count
sentiment_label,
positive,391776
neutral,148318
negative,102810



Kaggle validation sentiment distribution


,count
sentiment_label,
positive,48972
neutral,18539
negative,12852



Kaggle internal-test sentiment distribution


,count
sentiment_label,
positive,48973
neutral,18540
negative,12851


... (2 outputs left out)


Kaggle training aspect-label counts


,count
aspect,
career_opp,529639
work_life_balance,527824
comp_benefits,527649
senior_mgmt,523194
culture_values,496367
diversity_inclusion,104453



Bright external-test aspect-label counts


,count
aspect,
work_life_balance,431
culture_values,431
diversity_inclusion,431
comp_benefits,431
senior_mgmt,431
career_opp,143



## 14. Save cleaned outputs

Files are saved to `data/processed/`.

### Cleaned review-level data
- `kaggle_clean_reviews.csv`
- `brightdata_recent_clean_reviews.csv`

### Kaggle review-level splits
- `kaggle_reviews_train.csv`
- `kaggle_reviews_validation.csv`
- `kaggle_reviews_test.csv`

### Sentiment tables
- `kaggle_sentiment_train.csv`
- `kaggle_sentiment_validation.csv`
- `kaggle_sentiment_test.csv`
- `brightdata_sentiment_external_test.csv`

### Aspect tables
- `kaggle_aspect_train.csv`
- `kaggle_aspect_validation.csv`
- `kaggle_aspect_test.csv`
- `brightdata_aspect_external_test.csv`

Bright Data remains separate from Kaggle and is used only as the recent external test set / later RAG source.


In [43]:

OUTPUTS = {
    "kaggle_clean_reviews.csv":
        kaggle_clean,

    "brightdata_recent_clean_reviews.csv":
        bright_recent,

    "kaggle_reviews_train.csv":
        kaggle_train_reviews,

    "kaggle_reviews_validation.csv":
        kaggle_validation_reviews,

    "kaggle_reviews_test.csv":
        kaggle_test_reviews,

    "kaggle_sentiment_train.csv":
        kaggle_sentiment_train,

    "kaggle_sentiment_validation.csv":
        kaggle_sentiment_validation,

    "kaggle_sentiment_test.csv":
        kaggle_sentiment_test,

    "brightdata_sentiment_external_test.csv":
        bright_sentiment_external_test,

    "kaggle_aspect_train.csv":
        kaggle_aspect_train,

    "kaggle_aspect_validation.csv":
        kaggle_aspect_validation,

    "kaggle_aspect_test.csv":
        kaggle_aspect_test,

    "brightdata_aspect_external_test.csv":
        bright_aspect_external_test,
}

for filename, dataframe in OUTPUTS.items():
    path = PROCESSED_DIR / filename
    dataframe.to_csv(path, index=False)

    print(
        f"Saved {len(dataframe):,} rows -> {path}"
    )


Saved 803,631 rows -> data/processed/kaggle_clean_reviews.csv
Saved 431 rows -> data/processed/brightdata_recent_clean_reviews.csv
Saved 642,904 rows -> data/processed/kaggle_reviews_train.csv
Saved 80,363 rows -> data/processed/kaggle_reviews_validation.csv
Saved 80,364 rows -> data/processed/kaggle_reviews_test.csv
Saved 642,904 rows -> data/processed/kaggle_sentiment_train.csv
Saved 80,363 rows -> data/processed/kaggle_sentiment_validation.csv
Saved 80,364 rows -> data/processed/kaggle_sentiment_test.csv
Saved 431 rows -> data/processed/brightdata_sentiment_external_test.csv
Saved 2,709,126 rows -> data/processed/kaggle_aspect_train.csv
Saved 338,400 rows -> data/processed/kaggle_aspect_validation.csv
Saved 338,942 rows -> data/processed/kaggle_aspect_test.csv
Saved 2,298 rows -> data/processed/brightdata_aspect_external_test.csv



## Final output

The cleaned data are now organized for the modeling stage.

Recommended use:

- **Kaggle 80% training set** → LoRA/model fine-tuning
- **Kaggle 10% validation set** → model selection and tuning
- **Kaggle 10% internal test set** → final same-source evaluation
- **Bright Data 2022+** → recent external evaluation and later RAG demonstration

The Bright Data records should remain untouched during training and model-selection decisions.

> The Kaggle training split is large. For resource-limited LoRA experiments, a smaller **stratified sample should be drawn only from the training split**, while validation and test sets remain unchanged.
